In [ ]:
from pathlib import Path
import sys

helper_dir = Path.cwd().resolve() / "scripts"
if not (helper_dir / "workshop_helpers.py").is_file():
    raise RuntimeError("Start JupyterLab from the workshop folder.")

if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

from workshop_helpers import run_environment_check

run_environment_check()

# ROSCon Workshop: Stair-Climbing Policy — Evaluation

This notebook visualizes the generated terrain and compares the same Unitree G1 policy before and
after stair fine-tuning.

Hosted by **AMD and Robotec.ai**, the workshop runs on an **AMD Strix Halo mini-PC**. Evaluation
uses a fixed terrain grid, reset seed, command, and duration so that the checkpoint is the only
variable in the comparison.

## Evaluation goals

* inspect the collision geometry the robot actually encounters;
* quantify where the flat-ground baseline fails;
* connect aggregate results to visible behavior; and
* repeat the identical evaluation with the newly trained checkpoint.

## Setup

The infrastructure setup is documented in [`INSTALL.md`](INSTALL.md). Select the
**gslab ROSCon (ROCm 7.2)** kernel and run cells in order.

The first cell must print `PASS`. This notebook reads the trained policy from the fixed path
`logs/roscon_stairs/model_stairs.pt`; no timestamp or directory search is required.

In [ ]:
from pathlib import Path

import genesis as gs
import torch

import gslab.tasks  # registers every task  # noqa: F401
from gslab.tasks.registry import list_tasks, load_env_cfg, load_rl_cfg


def find_workshop_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INSTALL.md").is_file() and (candidate / "checkpoints").is_dir():
            return candidate
    raise RuntimeError("Could not find the workshop folder. Start Jupyter from that folder.")


WORKSHOP_ROOT = find_workshop_root(Path.cwd())
if not getattr(gs, "_initialized", False) or gs.backend != gs.amdgpu:
    raise RuntimeError("Run the first-cell infrastructure smoke test before continuing.")

DEVICE = "cuda"
GPU = torch.cuda.get_device_properties(0)

TASK = "Unitree-G1-Stairs-Easy"
BASELINE = WORKSHOP_ROOT / "checkpoints/g1_flat_baseline.pt"
TRAINED_PATH = WORKSHOP_ROOT / "logs/roscon_stairs/model_stairs.pt"
REFERENCE_PATH = WORKSHOP_ROOT / "checkpoints/g1_stairs_easy_trained.pt"

print(f"workshop  : {WORKSHOP_ROOT}")
print(f"accelerator: {GPU.name} ({GPU.total_memory / 2**30:.1f} GiB)")
print(f"PyTorch    : {torch.__version__}")
print(f"G1 tasks   : {[task for task in list_tasks() if 'G1' in task]}")
print(f"baseline   : {BASELINE.relative_to(WORKSHOP_ROOT)}")
print(f"trained    : {TRAINED_PATH.relative_to(WORKSHOP_ROOT)}")


In [ ]:
import sys

NOTEBOOK_DIR = WORKSHOP_ROOT / "scripts"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from workshop_helpers import EvaluationSession, clear_device_cache, require_checkpoint

## 1. Inspect the generated terrain

The task contains four terrain columns and six difficulty rows. Three columns begin as sampled
heightfields; `real_stairs` uses rigid boxes so Genesis sees actual vertical risers. The images
below show the collision geometry, including a close-up of the hardest stairs.

In [ ]:
from gslab.vis import render_terrain

terrain_cfg = load_env_cfg(TASK, play=True).scene.terrain.terrain_generator
terrain_frames = render_terrain(terrain_cfg, width=860, height=520)
clear_device_cache()


<details>
<summary><strong>Expected Genesis terrain warnings</strong></summary>

A generated heightfield is an open static surface, so Genesis may warn that it is not watertight
and may estimate unused inertial properties. It may also warn about SDF preprocessing for a large
mesh. These messages are expected here; convexifying the terrain would change the stairs we want
to evaluate.
</details>

## Build the evaluation scene

One scene is reused for every measurement and playback. It contains only eight robots; the
evaluator cycles that batch through all six difficulty levels and distributes it evenly across the
four terrain types. The model runner is constructed quietly, keeping the actor and critic
architecture dump out of the result cells. The overview camera stays fixed and frames the entire
terrain instead of following a robot.

In [ ]:
evaluation = EvaluationSession(task=TASK, num_envs=8, device=DEVICE)
print(evaluation.summary())


## 2. Evaluate the flat-ground baseline

The baseline is given a fixed `0.8 m/s` forward command for six seconds. Falls and forward travel
are aggregated by difficulty level across all four terrain types. The transposed table keeps the
levels in columns, making the failure boundary easy to scan.

In [ ]:
baseline_rows, baseline_overall = evaluation.evaluate(BASELINE)
evaluation.show_results(baseline_rows, baseline_overall, "before fine-tuning")


### Reference baseline result

| metric | level 0 | level 1 | level 2 | level 3 | level 4 | level 5 | all |
|---|---:|---:|---:|---:|---:|---:|---:|
| falls | 0% | 0% | 0% | 25% | 25% | 50% | **17%** |
| travel | 4.33 m | 4.71 m | 4.96 m | 4.28 m | 4.43 m | 3.35 m | **4.34 m** |

The easiest three levels are within the flat gait's existing capability. Failures appear sharply
from level 3 onward, which gives the terrain curriculum a clear progression to learn.

## 3. Watch the baseline fail

The 30-second playback uses the hardest rigid staircase, the same reset seed, and the same
forward command used for the trained policy. The fixed overview keeps the entire terrain in frame. Watch whether the feet lift before the first riser or react only after
contact, and how that late response affects torso balance.

In [ ]:
evaluation.watch(BASELINE, "flat-ground baseline", duration_s=30.0)


## 4. Evaluate the stair-trained policy

The primary path is the fixed checkpoint produced by the training notebook. If a workshop run is
interrupted, set `USE_SUPPLIED_REFERENCE = True` to use the bundled recovery checkpoint instead.
No timestamp or directory search is required.

In [ ]:
USE_SUPPLIED_REFERENCE = False
TRAINED = REFERENCE_PATH if USE_SUPPLIED_REFERENCE else TRAINED_PATH
TRAINED = require_checkpoint(TRAINED)

print(f"using: {TRAINED.relative_to(WORKSHOP_ROOT)}")


In [ ]:
trained_rows, trained_overall = evaluation.evaluate(TRAINED)
evaluation.show_results(trained_rows, trained_overall, "after fine-tuning")


### Reference fine-tuned result

A 200-update reference run reduced the fall rate through level 4 to zero; remaining failures were
concentrated on the hardest level.

| metric | level 0 | level 1 | level 2 | level 3 | level 4 | level 5 | all |
|---|---:|---:|---:|---:|---:|---:|---:|
| falls | 0% | 0% | 0% | 0% | 0% | 23% | **4%** |

Exact values can vary with stochastic PPO updates. The important comparison is the movement of the
failure boundary toward the top of the difficulty ladder.

#### Why travel can decrease after fine-tuning

The commanded forward speed is 0.8 m/s and each measurement lasts 6 seconds, so ideal command
tracking would cover 4.8 m. In the example run, the baseline travelled 4.41 m overall but fell in
20% of trials; the fine-tuned policy travelled 3.73 m and completed every trial without falling.

The fine-tuned policy has learned a stability–speed trade-off. It advances at about 0.62 m/s,
using more conservative steps, longer support phases, and enough foot motion to negotiate the
terrain. The baseline moves closer to the requested speed on easy ground, but that gait becomes
brittle as difficulty rises. Its travel measurement is also cut off at the first fall, so travel
alone is not a complete measure of policy quality.

The nearly constant fine-tuned distance across all six levels is a positive result: terrain
difficulty no longer causes a sharp loss of progress. Read **falls and travel together**—the
fine-tuned policy is more robust, but its remaining opportunity is better velocity tracking
without giving up that stability.


## 5. Watch the improved policy

Only the checkpoint changes. Look for the behavior behind the metrics: earlier foot lift, recovery
from unexpected contact, and a climbing posture that remains controlled across successive steps.

In [ ]:
evaluation.watch(TRAINED, "stair-trained policy", duration_s=30.0)


## Key takeaways

* Fixed terrain assignments expose where a policy fails instead of hiding difficulty inside one
  average score.
* Quantitative evaluation and playback answer different questions: how often it fails, and why.
* Fine-tuning should move the failure boundary toward harder terrain without sacrificing the easy
  levels.

> **ROS 2 hint:** If the actor is later wrapped in a controller, reproduce its observation
> ordering, normalization, and 50 Hz control rate exactly.

## Cleanup

Run this cell before opening another GPU-heavy notebook.

In [ ]:
if "evaluation" in globals() and evaluation is not None:
    evaluation.close()
evaluation = None
print("Evaluation scene released.")
